# English to Spanish Translator

## Colab Training Notebook

This notebook trains and evaluates the Transformer model in Google Colab.

Before running the notebook:
- switch Colab to a GPU runtime
- review the training settings before starting
- expect the OPUS download and preprocessing stages to take time on the full corpus mix


## 1. Clone Repository

This notebook can be uploaded to Colab by itself. The first code cell clones the project repository into `/content` if the project files are not already present.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/mathew-felix/english-spanish-translator.git"
REPO_DIR = Path("/content/english-spanish-translator")

def looks_like_repo(path: Path) -> bool:
    return (
        path.exists()
        and (path / ".git").exists()
        and (path / "source").exists()
        and (path / "run.py").exists()
    )

current_dir = Path.cwd()

if looks_like_repo(current_dir):
    project_dir = current_dir
elif looks_like_repo(REPO_DIR):
    project_dir = REPO_DIR
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    project_dir = REPO_DIR

os.chdir(project_dir)
print("Working directory:", Path.cwd())


## 2. Install Dependencies

Colab already includes a CUDA-enabled PyTorch build. This cell installs the remaining project dependencies without replacing the Colab PyTorch packages.


In [ ]:
from pathlib import Path

skip_packages = {"torch", "torchvision", "torchaudio"}
filtered_lines = []

for line in Path("requirements.txt").read_text().splitlines():
    stripped = line.strip()
    if not stripped or stripped.startswith("#"):
        continue
    package_name = stripped.split("==")[0]
    if package_name in skip_packages:
        continue
    filtered_lines.append(stripped)

Path("/tmp/colab-requirements.txt").write_text("\n".join(filtered_lines) + "\n")

!pip install -q -r /tmp/colab-requirements.txt

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 3. Optional W&B Configuration

Set your Weights & Biases API key here if you want live cloud tracking. If the key is left blank, training will still run and log locally in offline mode.


In [ ]:
import os

WANDB_API_KEY = ""
WANDB_PROJECT = "english-spanish-translator"
WANDB_ENTITY = ""

if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if os.getenv("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "online"
    print("W&B online logging enabled.")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("W&B API key not set. Training will log offline in this Colab session.")


## 4. Training Configuration

Set `QUICK_SANITY_RUN = True` for the first Colab pass. That mode downloads a smaller corpus subset, builds a small sampled train/test split, and runs a short training job to confirm the notebook works end to end.

For the full run on your RTX 6000 Pro setup, keep `QUICK_SANITY_RUN = False`. `MAX_SEQ_LENGTH=60` matches your stronger checkpoint setting, and `LEARNING_RATE=3e-4` is the safer full-run starting point.


In [ ]:
QUICK_SANITY_RUN = True

# Full training defaults
NUM_EPOCHS = 30
BATCH_SIZE = 256
MAX_SEQ_LENGTH = 60
LEARNING_RATE = 3e-4
PATIENCE = 5
WARMUP_STEPS = 4000
BEAM_WIDTH = 4
BLEU_EVAL_BATCHES = 5
OPEN_SUBTITLES_MAX_ROWS = 2_000_000
NUM_WORKERS = 4

# Sanity-run overrides
SANITY_CORPORA = ("Europarl", "News-Commentary", "TED2020")
SANITY_SAMPLE_FRACTION = 0.01
SANITY_NUM_EPOCHS = 1
SANITY_BATCH_SIZE = 64
SANITY_LEARNING_RATE = 2e-4
SANITY_BLEU_EVAL_BATCHES = 1

if QUICK_SANITY_RUN:
    DOWNLOAD_CORPORA = SANITY_CORPORA
    EFFECTIVE_NUM_EPOCHS = SANITY_NUM_EPOCHS
    EFFECTIVE_BATCH_SIZE = SANITY_BATCH_SIZE
    EFFECTIVE_LEARNING_RATE = SANITY_LEARNING_RATE
    EFFECTIVE_BLEU_EVAL_BATCHES = SANITY_BLEU_EVAL_BATCHES
    EFFECTIVE_OPEN_SUBTITLES_MAX_ROWS = 0
else:
    DOWNLOAD_CORPORA = ("Europarl", "News-Commentary", "TED2020", "OpenSubtitles")
    EFFECTIVE_NUM_EPOCHS = NUM_EPOCHS
    EFFECTIVE_BATCH_SIZE = BATCH_SIZE
    EFFECTIVE_LEARNING_RATE = LEARNING_RATE
    EFFECTIVE_BLEU_EVAL_BATCHES = BLEU_EVAL_BATCHES
    EFFECTIVE_OPEN_SUBTITLES_MAX_ROWS = OPEN_SUBTITLES_MAX_ROWS

print("Quick sanity run:", QUICK_SANITY_RUN)
print("Corpora:", DOWNLOAD_CORPORA)
print("Epochs:", EFFECTIVE_NUM_EPOCHS)
print("Batch size:", EFFECTIVE_BATCH_SIZE)
print("Learning rate:", EFFECTIVE_LEARNING_RATE)
print("OpenSubtitles max rows:", EFFECTIVE_OPEN_SUBTITLES_MAX_ROWS)


## 5. Download Dataset

This cell downloads the English-Spanish corpus mix from OPUS. In quick sanity mode it skips OpenSubtitles so the first Colab pass stays manageable.


In [ ]:
import source.DatasetDownload as dataset_download

dataset_download.OPUS_CORPORA = tuple(DOWNLOAD_CORPORA)
dataset_download.datasetDownload()

!du -sh data/raw
!find data/raw -maxdepth 2 -type f | head -n 20


## 6. Preprocess Dataset

This cell merges the selected corpora into one bilingual CSV, filters noisy sentence pairs, and writes `train.csv` and `test.csv`. In quick sanity mode it also samples a small fraction of the merged corpus before the split.


In [ ]:
import pandas as pd
import source.DatasetPreprocessing as dataset_preprocessing

dataset_preprocessing.CORPUS_LIMITS = {
    corpus: dataset_preprocessing.CORPUS_LIMITS[corpus]
    for corpus in DOWNLOAD_CORPORA
}

if "OpenSubtitles" in dataset_preprocessing.CORPUS_LIMITS:
    dataset_preprocessing.OPEN_SUBTITLES_MAX_ROWS = EFFECTIVE_OPEN_SUBTITLES_MAX_ROWS
    dataset_preprocessing.CORPUS_LIMITS["OpenSubtitles"] = EFFECTIVE_OPEN_SUBTITLES_MAX_ROWS

dataset_preprocessing.DatasetPreprocessing()

if QUICK_SANITY_RUN:
    sanity_path = "data/sanity_dataset.csv"
    dataset_preprocessing.SmallDataset(
        dataset_preprocessing.MERGED_DATASET_PATH,
        sanity_path,
        SANITY_SAMPLE_FRACTION,
    )
    dataset_preprocessing.Split_data(sanity_path)

for csv_path in ["data/english_spanish.csv", "data/train.csv", "data/test.csv"]:
    df = pd.read_csv(csv_path, nrows=5)
    print(f"\n{csv_path}")
    print(df.head())

!ls -lh data


## 7. Train Model

This cell keeps the project model architecture unchanged and only overrides the runtime training settings for Colab. In quick sanity mode it uses the lighter overrides defined above.


In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer
from source.Config import Config
from source.DatasetTranslation import TranslationDataset
from source.Model import Transformer
from source.Train import train_model, _encode_sentence

config = Config()
config.num_epochs = EFFECTIVE_NUM_EPOCHS
config.batch_size = EFFECTIVE_BATCH_SIZE
config.max_seq_length = MAX_SEQ_LENGTH
config.learning_rate = EFFECTIVE_LEARNING_RATE
config.patience = PATIENCE
config.warmup_steps = WARMUP_STEPS
config.bleu_eval_batches = EFFECTIVE_BLEU_EVAL_BATCHES
config.wandb_project = os.getenv("WANDB_PROJECT", config.wandb_project)
config.wandb_entity = os.getenv("WANDB_ENTITY") or None
config.wandb_mode = os.getenv("WANDB_MODE", config.wandb_mode)

torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

os.makedirs(config.tokenizer_path, exist_ok=True)

tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")
tokenizer.add_special_tokens(
    {
        "additional_special_tokens": [
            config.pad_token,
            config.unk_token,
            config.sos_token,
            config.eos_token,
        ]
    }
)

config.vocab_size = len(tokenizer)
config.pad_token_id = tokenizer.convert_tokens_to_ids(config.pad_token)
config.unk_token_id = tokenizer.convert_tokens_to_ids(config.unk_token)
config.sos_token_id = tokenizer.convert_tokens_to_ids(config.sos_token)
config.eos_token_id = tokenizer.convert_tokens_to_ids(config.eos_token)

tokenizer.save_pretrained(config.tokenizer_path)

train_dataset_full = TranslationDataset(
    csv_file=config.train_csv,
    tokenizer=tokenizer,
    sequence_length=config.max_seq_length,
    config=config,
)

train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size
split_generator = torch.Generator().manual_seed(config.seed)
train_dataset, val_dataset = torch.utils.data.random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=split_generator,
)

num_workers = NUM_WORKERS if torch.cuda.is_available() else 0
pin_memory = torch.cuda.is_available()

train_dataloader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

model = Transformer(config).to(config.device)
train_model(model, train_dataloader, val_dataloader, config, tokenizer)

checkpoint = torch.load(config.model_save_path, map_location=config.device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Training complete. Best checkpoint:", config.model_save_path)


## 8. Evaluate Model

This cell evaluates the best checkpoint against the test split created in the preprocessing step and writes `bleu_score_distribution.png`.


In [ ]:
from source.Evaluate import evaluate_model

test_dataset = TranslationDataset(
    csv_file=config.test_csv,
    tokenizer=tokenizer,
    sequence_length=config.max_seq_length,
    config=config,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

avg_bleu = evaluate_model(
    model,
    test_dataloader,
    config,
    tokenizer,
    max_seq_length=config.max_seq_length,
)

print("Final BLEU:", avg_bleu)


## 9. Example Translations

Use the trained checkpoint to translate a few sample English sentences.


In [ ]:
examples = [
    "How are you?",
    "Where is the nearest hospital?",
    "I need help with my homework.",
]

for text in examples:
    encoder_input = _encode_sentence(text, tokenizer, config)
    token_ids = model.generate(encoder_input, config, beam_width=BEAM_WIDTH)
    translation = tokenizer.decode(token_ids, skip_special_tokens=True)
    print(f"{text} -> {translation}")


## 10. Export Artifacts

This cell mounts Google Drive, collects the main training artifacts, zips them, and saves the archive into your Drive automatically.


In [ ]:
import os
import shutil
from datetime import datetime
from google.colab import drive

drive.mount("/content/drive")

drive_output_dir = "/content/drive/MyDrive/english-spanish-translator-artifacts"
os.makedirs(drive_output_dir, exist_ok=True)

artifact_paths = [
    "best_model.pth",
    "loss_plot.png",
    "bleu_score_distribution.png",
    "data/train.csv",
    "data/test.csv",
    "data/tokenizer",
]

if os.path.isdir("wandb"):
    artifact_paths.append("wandb")

staging_dir = "/tmp/english_spanish_translator_export"
if os.path.exists(staging_dir):
    shutil.rmtree(staging_dir)
os.makedirs(staging_dir, exist_ok=True)

for artifact in artifact_paths:
    if not os.path.exists(artifact):
        print("Missing:", artifact)
        continue

    destination = os.path.join(staging_dir, os.path.basename(artifact))
    if os.path.isdir(artifact):
        shutil.copytree(artifact, destination)
    else:
        shutil.copy2(artifact, destination)
    print("Added:", artifact)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_base = os.path.join(drive_output_dir, f"english_spanish_translator_{timestamp}")
archive_path = shutil.make_archive(archive_base, "zip", staging_dir)

print("Saved archive to:", archive_path)
